# nb07a — One-Stage Anchor-Free: FCOS Baseline (F0)

**Purpose.** Train `fcos_resnet50_fpn` (COCO-pretrained, 9-class head) under the locked protocol (letterbox-1280, SGD lr=5e-3, 1 epoch, clip-disjoint split).

**Why FCOS.** Anchor-free detector: every location inside an object (by center sampling with radius 1.5× stride) predicts class + 4 box-edge distances + centerness. No anchors.

**Config (F0 baseline):**
- `center_sampling_radius = 1.5` (torchvision default)
- `iou_loss_type = "giou"` (default)

**This notebook produces:**
- `fcos_F0.pth` — checkpoint (upload to Kaggle dataset `fcos-1`)
- `val_preds_fcos_F0.json`, `metrics_fcos_F0.json`, `speed_fcos_F0.json`, `train_log_fcos_F0.json`
- W&B run `fcos_F0`

**No submission in this notebook.** F1 will be trained in nb07b, and submission is generated there from whichever variant wins.

In [1]:
import sys, json, time, shutil, os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.transforms import functional as TF

# ---- Paths -----------------------------------------------------------------
COMP_DIR    = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR = Path("/kaggle/input/datasets/suchok/outputs")
WORK_DIR    = Path("/kaggle/working")

assert COMP_DIR.exists(),    f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(), f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(), f"missing outputs: {OUTPUTS_DIR}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ---- common.py --------------------------------------------------------------
sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES,
    load_splits, index_split, filter_by_clips,
    VisDroneDetectionDataset, detection_collate,
    build_fullval_gt,
    save_checkpoint, save_metrics,
    IMAGENET_MEAN, IMAGENET_STD,
)

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

# Speed flags
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

# ---- Locked protocol --------------------------------------------------------
chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
train_index = filter_by_clips(index_split(COMP_DIR / "train"), train_clips)
val_index   = filter_by_clips(index_split(COMP_DIR / "val"),   val_clips)
print(f"train frames: {len(train_index):,} | val frames: {len(val_index):,}")

# ---- Full-val GT (cached from previous notebooks) ---------------------------
gt_fullval_ignore = WORK_DIR / "gt_fullval_ignore.json"
for src in [Path("/kaggle/input/datasets/suchok/retinanetr-1"),
            Path("/kaggle/input/datasets/suchok/retinanet-1"),
            Path("/kaggle/input/datasets/suchok/outputs")]:
    cand = src / "gt_fullval_ignore.json"
    if cand.exists() and not gt_fullval_ignore.exists():
        shutil.copy(cand, gt_fullval_ignore)
        print(f"copied GT from {cand}")
        break
if not gt_fullval_ignore.exists():
    build_fullval_gt(val_index, str(gt_fullval_ignore), include_ignore=True)
    print("built GT")

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
train frames: 24,201 | val frames: 2,846
copied GT from /kaggle/input/datasets/suchok/retinanetr-1/gt_fullval_ignore.json


In [2]:
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
WANDB_PROJECT = "visdrone-det"
print("W&B ready. project:", WANDB_PROJECT)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready. project: visdrone-det


## 1. FCOS builder (F0 baseline)

FCOS outputs per location: `(class_probs, 4 distances l,t,r,b, centerness)`. Head swap: 91-class → 9-class (8 VisDrone + background). Center sampling radius 1.5× stride (torchvision default).

In [3]:
def build_fcos(center_sampling_radius=1.5, iou_loss_type="giou",
               score_thresh=0.05, nms_thresh=0.5):
    m = fcos_resnet50_fpn(
        weights=None, weights_backbone=None,
        min_size=SIZE, max_size=SIZE,
        score_thresh=score_thresh,
        nms_thresh=nms_thresh,
        detections_per_img=500,
        center_sampling_radius=center_sampling_radius,
    )
    in_channels = m.head.classification_head.cls_logits.in_channels
    num_anchors = m.head.classification_head.num_anchors
    m.head.classification_head = FCOSClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=lambda c: torch.nn.GroupNorm(32, c),
    )
    m.iou_loss_type = iou_loss_type
    print(f"FCOS built: center_sampling_radius={center_sampling_radius}, "
          f"iou_loss_type={iou_loss_type}, num_anchors={num_anchors}")
    return m


F0_CFG = {"center_sampling_radius": 1.5, "iou_loss_type": "giou"}
model = build_fcos(**F0_CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params/1e6:.2f}M")

FCOS built: center_sampling_radius=1.5, iou_loss_type=giou, num_anchors=1
trainable params: 32.13M


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    VisDroneDetectionDataset(train_index, STRAT, SIZE, is_train=False),
    batch_size=2, shuffle=True, num_workers=2,
    collate_fn=detection_collate, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    VisDroneDetectionDataset(val_index, STRAT, SIZE, is_train=False),
    batch_size=2, shuffle=False, num_workers=2,
    collate_fn=detection_collate, pin_memory=True,
)
print(f"train batches: {len(train_loader):,} | val batches: {len(val_loader):,}")

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=5e-3, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Smoke test
model.train()
imgs, tgts = next(iter(train_loader))
imgs = [i.to(DEVICE) for i in imgs]
tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
with torch.no_grad():
    ld = model(imgs, tgts)
print("smoke loss:", {k: round(float(v.detach()), 4) for k, v in ld.items()})

train batches: 12,100 | val batches: 1,423
smoke loss: {'classification': 1.2218, 'bbox_regression': 1.0, 'bbox_ctrness': 0.7543}


In [5]:
def train_one_epoch(model, loader, optimizer, scheduler,
                     device, run_name, wandb_run,
                     log_every=100, wandb_log_every=20):
    model.train()
    running = {"loss": 0.0, "loss_cls": 0.0, "loss_box": 0.0, "loss_centerness": 0.0}
    n_batches = 0
    t_start = time.time()

    for step, (images, targets) in enumerate(tqdm(loader, desc=f"{run_name}")):
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()}
                   for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        losses.backward()
        optimizer.step()

        # NOTE: FCOS returns 'bbox_ctrness' (not 'centerness'). Use a tensor
        # default so .detach() is always valid even if the key is absent.
        zero = torch.tensor(0.0, device=device)
        cls_loss   = loss_dict.get("classification", zero)
        box_loss   = loss_dict.get("bbox_regression", zero)
        ctr_loss   = loss_dict.get("bbox_ctrness", zero)

        running["loss"]           += float(losses.detach())
        running["loss_cls"]       += float(cls_loss.detach())
        running["loss_box"]       += float(box_loss.detach())
        running["loss_centerness"] += float(ctr_loss.detach())
        n_batches += 1

        if step % log_every == 0 and step > 0:
            elapsed = time.time() - t_start
            spb = elapsed / (step + 1)
            remaining = (len(loader) - step - 1) * spb
            print(f"[{run_name} {step}/{len(loader)}] "
                  f"elapsed={elapsed/60:.1f}min {spb:.2f}s/batch "
                  f"ETA={remaining/3600:.2f}h loss={float(losses.detach()):.4f}",
                  flush=True)

        if step % wandb_log_every == 0:
            wandb_run.log({
                "train/loss":           float(losses.detach()),
                "train/loss_cls":       float(cls_loss.detach()),
                "train/loss_box":       float(box_loss.detach()),
                "train/loss_centerness": float(ctr_loss.detach()),
                "train/step":           step,
            })

    scheduler.step()
    mean = {k: v / max(1, n_batches) for k, v in running.items()}
    return mean, time.time() - t_start

In [6]:
@torch.no_grad()
def run_detector_eval(detector, image_rgb, score_thr=0.001, device=None):
    if device is None:
        device = next(detector.parameters()).device
    detector.eval()
    tensor = TF.to_tensor(image_rgb)
    tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    out = detector([tensor.to(device)])[0]
    keep = out["scores"] >= score_thr
    return (out["boxes"][keep].cpu().numpy(),
            out["labels"][keep].cpu().numpy(),
            out["scores"][keep].cpu().numpy())


def predict_with_strategy_eval(detector, image_path, strategy, size,
                                score_thr=0.001, device=None):
    from common import apply_strategy, boxes_to_original
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    h0, w0 = img.shape[:2]
    inp, meta = apply_strategy(img, strategy, size, seed_key=image_path)
    t0 = time.time()
    boxes, labels, scores = run_detector_eval(detector, inp,
                                               score_thr=score_thr, device=device)
    dt = time.time() - t0
    boxes_orig = boxes_to_original(boxes, meta, strategy, w0, h0)
    return boxes_orig, labels, scores, dt, meta


def predictions_to_coco_eval(df_index, detector, strategy, size, out_json,
                              score_thr=0.001, device=None):
    preds = []
    id_map = {row["image_path"]: i for i, (_, row) in enumerate(
        df_index.iterrows(), start=1)}
    valid_cat_ids = set(range(1, NUM_CLASSES + 1))
    for _, row in tqdm(df_index.iterrows(), total=len(df_index),
                       desc=f"pred {strategy}-{size}"):
        boxes, labels, scores, _, _ = predict_with_strategy_eval(
            detector, row["image_path"], strategy, size,
            score_thr=score_thr, device=device)
        img_id = id_map[row["image_path"]]
        for (x1, y1, x2, y2), l, s in zip(boxes, labels, scores):
            if int(l) not in valid_cat_ids:
                continue
            preds.append({
                "image_id": int(img_id),
                "category_id": int(l),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(s),
            })
    with open(out_json, "w") as f:
        json.dump(preds, f)
    print(f"wrote {len(preds):,} predictions -> {out_json}")


def coco_eval_summary_safe(gt_json, dt_json):
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(gt_json)
        if os.path.getsize(dt_json) == 0 or json.load(open(dt_json)) == []:
            return {k: 0.0 for k in ["mAP", "AP50", "AP75",
                                     "AP_small", "AP_medium", "AP_large"]}
        coco_dt = coco_gt.loadRes(dt_json)
        e = COCOeval(coco_gt, coco_dt, "bbox")
        e.evaluate(); e.accumulate(); e.summarize()
    return {
        "mAP":       float(e.stats[0]),
        "AP50":      float(e.stats[1]),
        "AP75":      float(e.stats[2]),
        "AP_small":  float(e.stats[3]),
        "AP_medium": float(e.stats[4]),
        "AP_large":  float(e.stats[5]),
    }

In [7]:
run = wandb.init(
    project=WANDB_PROJECT, name="fcos_F0",
    config=dict(model="fcos_resnet50_fpn", variant="FCOS_F0",
                strategy=STRAT, size=SIZE, epochs=1, lr=5e-3, batch_size=2,
                **F0_CFG, seed=SEED),
    reinit=True,
)

t0 = time.time()
train_stats, train_seconds = train_one_epoch(
    model, train_loader, optimizer, scheduler, DEVICE,
    run_name="F0", wandb_run=run, log_every=100, wandb_log_every=20,
)
print(f"\ntrain stats: {train_stats}")
print(f"train wall time: {train_seconds/3600:.2f} h")

save_checkpoint(model, WORK_DIR / "fcos_F0.pth")
with open(WORK_DIR / "train_log_fcos_F0.json", "w") as f:
    json.dump({"epoch": 1, "run": "F0",
               "mean_loss":     train_stats["loss"],
               "mean_loss_cls": train_stats["loss_cls"],
               "mean_loss_box": train_stats["loss_box"],
               "mean_loss_centerness": train_stats["loss_centerness"],
               "wall_seconds":  train_seconds,
               "config":        F0_CFG}, f, indent=2)
wandb.log({"train/epoch_loss": train_stats["loss"],
           "train/epoch_time_h": train_seconds/3600})
print("checkpoint + train_log saved")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261107_200927-ibval1aj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fcos_F0
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/ibval1aj


F0:   0%|          | 0/12100 [00:00<?, ?it/s]

[F0 100/12100] elapsed=2.4min 1.41s/batch ETA=4.71h loss=2.5435
[F0 200/12100] elapsed=4.6min 1.38s/batch ETA=4.57h loss=2.1840
[F0 300/12100] elapsed=7.0min 1.39s/batch ETA=4.54h loss=1.9932
[F0 400/12100] elapsed=9.3min 1.39s/batch ETA=4.52h loss=1.9361
[F0 500/12100] elapsed=11.6min 1.39s/batch ETA=4.48h loss=1.9052
[F0 600/12100] elapsed=14.0min 1.39s/batch ETA=4.45h loss=1.8839
[F0 700/12100] elapsed=16.3min 1.39s/batch ETA=4.41h loss=1.8539
[F0 800/12100] elapsed=18.6min 1.39s/batch ETA=4.38h loss=1.8894
[F0 900/12100] elapsed=20.9min 1.40s/batch ETA=4.34h loss=1.8194
[F0 1000/12100] elapsed=23.3min 1.40s/batch ETA=4.30h loss=1.9240
[F0 1100/12100] elapsed=25.6min 1.40s/batch ETA=4.27h loss=1.8751
[F0 1200/12100] elapsed=28.0min 1.40s/batch ETA=4.23h loss=1.8342
[F0 1300/12100] elapsed=30.3min 1.40s/batch ETA=4.19h loss=1.8117
[F0 1400/12100] elapsed=32.6min 1.40s/batch ETA=4.15h loss=1.8673
[F0 1500/12100] elapsed=35.0min 1.40s/batch ETA=4.11h loss=1.7113
[F0 1600/12100] elapsed

In [8]:
torch.cuda.empty_cache()
model.eval()
preds_path = WORK_DIR / "val_preds_fcos_F0.json"
t0 = time.time()
predictions_to_coco_eval(val_index, model, STRAT, SIZE, str(preds_path),
                         score_thr=0.001, device=DEVICE)
print(f"inference: {(time.time()-t0)/60:.2f} min")

metrics = coco_eval_summary_safe(str(gt_fullval_ignore), str(preds_path))
print("=" * 60)
print("F0 — full val:")
for k, v in metrics.items():
    print(f"  {k:10s}: {v:.4f}")
save_metrics(metrics, WORK_DIR / "metrics_fcos_F0.json")
wandb.log({f"val/{k}": v for k, v in metrics.items()})

pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

wrote 1,422,989 predictions -> /kaggle/working/val_preds_fcos_F0.json
inference: 17.06 min
F0 — full val:
  mAP       : 0.0258
  AP50      : 0.0503
  AP75      : 0.0252
  AP_small  : 0.0089
  AP_medium : 0.0202
  AP_large  : 0.0593


In [9]:
model.eval()
if DEVICE == "cuda":
    torch.cuda.synchronize()
for i in range(5):
    _ = predict_with_strategy_eval(model, val_index.iloc[i]["image_path"],
                                    STRAT, SIZE, score_thr=0.5, device=DEVICE)
times = []
for i in range(50):
    row = val_index.iloc[(5 + i) % len(val_index)]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    _ = predict_with_strategy_eval(model, row["image_path"], STRAT, SIZE,
                                    score_thr=0.5, device=DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
mean_s = float(np.mean(times))
speed = {"device": str(DEVICE),
         "gpu_ms_per_image": mean_s * 1000,
         "gpu_fps":          1.0 / mean_s,
         "n_images": 50, "score_thr": 0.5}
print("speed:", json.dumps(speed, indent=2))
with open(WORK_DIR / "speed_fcos_F0.json", "w") as f:
    json.dump(speed, f, indent=2)
wandb.log({"speed/gpu_fps": speed["gpu_fps"],
           "speed/gpu_ms":  speed["gpu_ms_per_image"]})

speed: {
  "device": "cuda",
  "gpu_ms_per_image": 273.99970531463623,
  "gpu_fps": 3.649638961661259,
  "n_images": 50,
  "score_thr": 0.5
}


In [10]:
print("=" * 70)
print("SESSION 1 COMPLETE — FCOS F0 trained and evaluated")
print("=" * 70)
print()
print(f"| nb07a | FCOS F0 | center_sampling=1.5, GIoU | "
      f"{metrics['mAP']:.4f} | {metrics['AP50']:.4f} | "
      f"{metrics['AP_small']:.4f} | {speed['gpu_fps']:.2f} FPS |")
print()
print("Artifacts in /kaggle/working/:")
for p in sorted(WORK_DIR.glob("*fcos_F0*")):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
print()
print("NEXT STEPS:")
print("  1. Download ALL fcos_F0* artifacts from /kaggle/working/")
print("  2. Upload as Kaggle dataset named 'fcos-1'")
print("  3. Open nb07b, add 'fcos-1' as input, train F1")
print("=" * 70)

run.finish()
print("W&B run finished:", run.url)

wandb: updating run metadata


SESSION 1 COMPLETE — FCOS F0 trained and evaluated

| nb07a | FCOS F0 | center_sampling=1.5, GIoU | 0.0258 | 0.0503 | 0.0089 | 3.65 FPS |

Artifacts in /kaggle/working/:
  fcos_F0.pth  (128.9 MB)
  metrics_fcos_F0.json  (0.0 MB)
  speed_fcos_F0.json  (0.0 MB)
  train_log_fcos_F0.json  (0.0 MB)
  val_preds_fcos_F0.json  (214.2 MB)

NEXT STEPS:
  1. Download ALL fcos_F0* artifacts from /kaggle/working/
  2. Upload as Kaggle dataset named 'fcos-1'
  3. Open nb07b, add 'fcos-1' as input, train F1


wandb: 
wandb: Run history:
wandb:         speed/gpu_fps ▁
wandb:          speed/gpu_ms ▁
wandb:      train/epoch_loss ▁
wandb:    train/epoch_time_h ▁
wandb:            train/loss ▅▆▅▅▆▅▅█▅▅▄▄▃▃▂▄▂▃▃▁▃▃▂▃▃▂▂▂▃▁▂▁▂▂▂▁▂▂▂▂
wandb:        train/loss_box █▆▇█▆▆▆▅▆▅▄▆▅▅▆▄▆▅▄▃▅▄▄▃▃▃▅▃▃▃▄▃▂▃▄▃▃▄▃▁
wandb: train/loss_centerness ▇▆▅▆█▇▄▅▅▆▃▇▄▃▂▁▄▄▄▃▄▅▅▅▄▄▄▄▃▃▂▃▂▆▂▁▂▃▄▁
wandb:        train/loss_cls ▄▄▄▄▅▄▄▅▅▄▅▄█▅▃▂▄▂▃▂▂▃▁▃▁▂▃▂▁▂▂▂▂▂▂▁▁▂▂▂
wandb:            train/step ▁▁▁▁▁▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▇▇▇▇▇▇▇▇▇█████
wandb:              val/AP50 ▁
wandb:                    +5 ...
wandb: 
wandb: Run summary:
wandb:         speed/gpu_fps 3.64964
wandb:          speed/gpu_ms 273.99971
wandb:      train/epoch_loss 1.58102
wandb:    train/epoch_time_h 4.72268
wandb:            train/loss 1.4757
wandb:        train/loss_box 0.39047
wandb: train/loss_centerness 0.6499
wandb:        train/loss_cls 0.43533
wandb:            train/step 12080
wandb:              val/AP50 0.05029
wandb:                    +5 ...
wandb

W&B run finished: https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/ibval1aj
